# Credit Card Fraud Detection

## Internship Task 2

This project uses machine learning to detect fraudulent credit card transactions.

Workflow:
- Load and explore the training and testing datasets
- Preprocess transaction data
- Engineer useful time-based features
- Handle class imbalance
- Train Logistic Regression and Random Forest models
- Evaluate with accuracy, precision, recall, F1-score, ROC-AUC and PR-AUC
- Compare models
- Generate fraud predictions


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

print("Libraries imported successfully!")


## 2. Load the Dataset

The notebook expects `fraudTrain.csv` and `fraudTest.csv` in the same folder as this notebook.


In [ ]:
train_data = pd.read_csv("fraudTrain.csv")
test_data = pd.read_csv("fraudTest.csv")

print("Training data shape:", train_data.shape)
print("Testing data shape:", test_data.shape)
display(train_data.head())


## 3. Explore the Dataset

We inspect columns, data types, missing values, duplicates, and the fraud/non-fraud distribution.


In [ ]:
print("Columns:")
print(train_data.columns.tolist())

print("\nMissing values:")
display(train_data.isnull().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", train_data.duplicated().sum())

print("\nFraud distribution:")
display(train_data["is_fraud"].value_counts())
display(train_data["is_fraud"].value_counts(normalize=True).mul(100).round(4))


In [ ]:
train_data["is_fraud"].value_counts().sort_index().plot(kind="bar", figsize=(7,4))
plt.title("Fraud vs Genuine Transactions")
plt.xlabel("is_fraud (0 = Genuine, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Feature Engineering

Transaction timestamps are converted into year, month, day, hour and day-of-week features. Identifier fields such as card number and transaction number are removed to reduce unnecessary complexity and leakage risk.


In [ ]:
def prepare_features(df):
    data = df.copy()

    if "trans_date_trans_time" in data.columns:
        dt = pd.to_datetime(data["trans_date_trans_time"], errors="coerce")
        data["trans_year"] = dt.dt.year
        data["trans_month"] = dt.dt.month
        data["trans_day"] = dt.dt.day
        data["trans_hour"] = dt.dt.hour
        data["trans_dayofweek"] = dt.dt.dayofweek
        data = data.drop(columns=["trans_date_trans_time"])

    drop_cols = ["is_fraud", "trans_num", "cc_num", "Unnamed: 0"]
    data = data.drop(columns=[c for c in drop_cols if c in data.columns])

    return data

X = prepare_features(train_data)
y = train_data["is_fraud"].astype(int)
X_test_external = prepare_features(test_data)

print("Prepared training shape:", X.shape)
print("Prepared test shape:", X_test_external.shape)


## 5. Train-Validation Split

The training data is split into training and validation sets. Stratification preserves the minority fraud class proportion.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Fraud rate:", round(y_train.mean() * 100, 4), "%")


## 6. Data Preprocessing

Numerical features are median-imputed and standardized. Categorical features are imputed and one-hot encoded.


In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## 7. Logistic Regression

Logistic Regression is used as a baseline model. `class_weight="balanced"` gives additional importance to the minority fraud class.


In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)
print("Logistic Regression training completed!")


In [ ]:
lr_pred = logistic_model.predict(X_val)
lr_prob = logistic_model.predict_proba(X_val)[:, 1]

print("Logistic Regression Results")
print("Accuracy :", accuracy_score(y_val, lr_pred))
print("Precision:", precision_score(y_val, lr_pred, zero_division=0))
print("Recall   :", recall_score(y_val, lr_pred, zero_division=0))
print("F1-score :", f1_score(y_val, lr_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_val, lr_prob))
print("PR-AUC   :", average_precision_score(y_val, lr_prob))

print("\nClassification Report:")
print(classification_report(y_val, lr_pred, digits=4, zero_division=0))


### 7.1 Logistic Regression Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val, lr_pred,
    display_labels=["Genuine", "Fraud"]
)
plt.title("Logistic Regression - Confusion Matrix")
plt.tight_layout()
plt.show()


## 8. Random Forest

Random Forest is evaluated as a second classification model. A sample is used for training when the dataset is very large so that the notebook remains practical on a normal laptop.


In [ ]:
rf_sample_size = min(200000, len(X_train))

if len(X_train) > rf_sample_size:
    sample_idx = train_data.loc[X_train.index].sample(
        n=rf_sample_size,
        random_state=42,
        stratify=y_train
    ).index
    X_rf = X_train.loc[sample_idx]
    y_rf = y_train.loc[sample_idx]
else:
    X_rf = X_train
    y_rf = y_train

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_rf, y_rf)
print("Random Forest training completed!")


In [ ]:
rf_pred = rf_model.predict(X_val)
rf_prob = rf_model.predict_proba(X_val)[:, 1]

print("Random Forest Results")
print("Accuracy :", accuracy_score(y_val, rf_pred))
print("Precision:", precision_score(y_val, rf_pred, zero_division=0))
print("Recall   :", recall_score(y_val, rf_pred, zero_division=0))
print("F1-score :", f1_score(y_val, rf_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_val, rf_prob))
print("PR-AUC   :", average_precision_score(y_val, rf_prob))


## 9. Model Comparison

Because fraud is usually a highly imbalanced classification problem, accuracy alone is not sufficient. Precision, recall, F1-score, ROC-AUC and PR-AUC are compared.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_val, lr_pred),
        accuracy_score(y_val, rf_pred)
    ],
    "Precision": [
        precision_score(y_val, lr_pred, zero_division=0),
        precision_score(y_val, rf_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_val, lr_pred, zero_division=0),
        recall_score(y_val, rf_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_val, lr_pred, zero_division=0),
        f1_score(y_val, rf_pred, zero_division=0)
    ],
    "ROC_AUC": [
        roc_auc_score(y_val, lr_prob),
        roc_auc_score(y_val, rf_prob)
    ],
    "PR_AUC": [
        average_precision_score(y_val, lr_prob),
        average_precision_score(y_val, rf_prob)
    ]
})

display(comparison.round(4))

best_model_name = comparison.sort_values("PR_AUC", ascending=False).iloc[0]["Model"]
best_model = rf_model if best_model_name == "Random Forest" else logistic_model

print("Selected final model:", best_model_name)


## 10. Final Test Predictions

The selected model is applied to `fraudTest.csv`, which was kept separate from model training.


In [ ]:
test_pred = best_model.predict(X_test_external)
test_prob = best_model.predict_proba(X_test_external)[:, 1]

print("Predictions completed!")
print("Number of test predictions:", len(test_pred))
print("Predicted fraud transactions:", int(test_pred.sum()))
print("Predicted genuine transactions:", int((test_pred == 0).sum()))


### 10.1 Final Test Evaluation

If `fraudTest.csv` contains the `is_fraud` target, final metrics are calculated. If it does not, predictions are generated but accuracy cannot be calculated without ground-truth labels.


In [ ]:
if "is_fraud" in test_data.columns:
    y_test_external = test_data["is_fraud"].astype(int)

    print("Final Test Accuracy :", accuracy_score(y_test_external, test_pred))
    print("Final Test Precision:", precision_score(y_test_external, test_pred, zero_division=0))
    print("Final Test Recall   :", recall_score(y_test_external, test_pred, zero_division=0))
    print("Final Test F1-score :", f1_score(y_test_external, test_pred, zero_division=0))
    print("Final Test ROC-AUC  :", roc_auc_score(y_test_external, test_prob))
else:
    print("No is_fraud column found in fraudTest.csv.")
    print("Ground-truth accuracy cannot be calculated from this file alone.")


## 11. Save Predictions

In [ ]:
predictions = pd.DataFrame({
    "prediction": test_pred
})

predictions.to_csv("fraud_predictions.csv", index=False)

print("Predictions saved to fraud_predictions.csv")
display(predictions.head(10))


## 12. New Transaction Prediction

The final model can also classify an individual transaction. The example below uses the first available test transaction as a demonstration input.


In [ ]:
new_transaction = X_test_external.iloc[[0]].copy()

new_prediction = best_model.predict(new_transaction)[0]
new_probability = best_model.predict_proba(new_transaction)[0, 1]

print("Predicted class:", "FRAUD" if new_prediction == 1 else "GENUINE")
print("Fraud probability:", round(new_probability, 4))


## 13. Conclusion

A machine learning system was developed to detect fraudulent credit card transactions. The project included data exploration, timestamp feature engineering, categorical encoding, numerical scaling, class-imbalance handling, model training, evaluation, and final prediction.

Logistic Regression and Random Forest were compared using multiple classification metrics. PR-AUC was emphasized because fraud detection is an imbalanced classification problem.

The selected model was then used to generate predictions for the separate test dataset and to demonstrate fraud prediction for a new transaction.


## 14. Final Project Summary

**Project:** Credit Card Transaction Fraud Detection

**Problem:** Binary classification

**Target:** `is_fraud`

**Models:** Logistic Regression and Random Forest

**Main techniques:** Feature engineering, one-hot encoding, scaling, class weighting, confusion matrix, precision, recall, F1-score, ROC-AUC and PR-AUC.

**Final output:** Genuine or Fraud prediction.
